# Biodiversity Habitat Mapping
**NB 17** | Borneo, Malaysia | Sentinel-2 | DINOv3 SAT + K-means (unsupervised)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.foundation.dinov3 import DINOv3Backbone, get_transform


In [ ]:
BBOX       = (117.0, 4.5, 117.5, 5.0)
DATE_RANGE = ('2024-01-01', '2024-06-30')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 20,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32650', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 6,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'borneo_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'foundation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# DINOv3 SAT spatial feature extraction on preprocessed scene
if PREPROCESSED.exists():
    # ALWAYS use get_transform() — SAT model uses different stats than ImageNet
    backbone  = DINOv3Backbone('dinov3_vitl16_sat')
    transform = get_transform('dinov3_vitl16_sat')   # SAT_MEAN/SAT_STD auto-selected
    features  = backbone.extract_features(str(PREPROCESSED))   # (H_patches, W_patches, 1024)
    attention = backbone.get_attention_maps(str(PREPROCESSED))  # saliency map
    print(f"DINOv3 spatial features: {features.shape}")
    print(f"Attention map          : {attention.shape}")
    print(f"Embedding dim          : {features.shape[-1]} (SAT-493M pre-training)")


In [ ]:
# K-means clustering on DINOv3 features → habitat clusters (no labels needed)
import numpy as np
import rasterio
from sklearn.cluster import KMeans

N_HABITATS = 8   # number of habitat types to discover
HABITAT_MAP = DATA_DIR / 'habitat_clusters.tif'
PREDICTION  = HABITAT_MAP

if 'features' in dir() and features is not None:
    H, W, D = features.shape
    X        = features.reshape(-1, D)
    kmeans   = KMeans(n_clusters=N_HABITATS, random_state=42, n_init=10)
    labels   = kmeans.fit_predict(X).reshape(H, W).astype(np.uint8)

    # Upsample labels to match preprocessed raster spatial extent
    with rasterio.open(str(PREPROCESSED)) as src:
        prof = src.profile.copy()
        rH, rW = src.height, src.width
    labels_full = np.kron(labels, np.ones((rH//H, rW//W), dtype=np.uint8))[:rH,:rW]

    prof.update(count=1, dtype='uint8', compress='lzw', nodata=255)
    with rasterio.open(str(HABITAT_MAP), 'w', **prof) as dst:
        dst.write(labels_full[np.newaxis])
    print(f"Habitat clusters: {N_HABITATS}  saved to {HABITAT_MAP}")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=50, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=0, min_area_m2=500.0, simplify_tolerance=0.5)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


In [ ]:
# Spectral characterisation of each habitat cluster
HABITAT_MAP = DATA_DIR / 'habitat_clusters.tif'
if HABITAT_MAP.exists() and PREPROCESSED.exists():
    stats = client.postprocess.class_statistics(str(HABITAT_MAP))
    zonal = client.postprocess.zonal_statistics(str(PREPROCESSED), str(DATA_DIR/'output.geojson'),
        output_path=str(DATA_DIR/'habitat_stats.geojson'), stats=['mean','std'])
    print("Habitat type statistics:")
    for cls, info in sorted(stats.items()):
        print(f"  Cluster {cls}: {info['area_ha']:.1f} ha ({info['pct']:.1f}%)")


**NB17 — Biodiversity** complete.
- Study area: Borneo Malaysia
- Sensor: Sentinel-2 6-band
- Model: DINOv3 SAT-493M + K-means unsupervised
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG